# Session 3: off-centre hits, a time-stepping simulation and a crash estimate
**A. Real-world problem.** Two billiard balls hit off-centre: the first one is deflected
and the second one leaves at an angle. How do the speeds and angles depend on how far off-centre
the hit is, and can we trust the formulas? We check them against a simulation in which the
discs push on each other with a stiff spring. Then we use the same ideas on a car hitting a wall.

Educational model: smooth (frictionless) discs, no spin, instant collision in the formulas and a linear
spring in the simulation. Not a tool for real crash analysis.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the contact direction decides everything
Disc 1 moves along +x. Disc 2 sits a distance b (the impact parameter) above that line. At the moment
of contact the line from the centre of disc 1 to the centre of disc 2 has the angle phi with
sin(phi) = b / (R1 + R2). Smooth discs push only along that line (the unit vector n), so
only the part of the relative velocity along n changes:

    v_n = (v1 - v2) . n
    v1' = v1 - m2/(m1 + m2) (1 + e) v_n n
    v2' = v2 + m1/(m1 + m2) (1 + e) v_n n

The part of the velocity across n (along the contact surface) does not change at all.

In [ ]:
def disc_collision(m1, m2, v1, v2, b, r1, r2, e=1.0):
    """Velocities (v1', v2') of two smooth discs after the collision. v1 and v2 are length-2 arrays.

    Disc 2 is offset by b (in the +y direction) from the line of motion of disc 1.
    """
    if m1 <= 0 or m2 <= 0 or r1 <= 0 or r2 <= 0:
        raise ValueError("masses and radii must be positive")
    if not 0 <= e <= 1:
        raise ValueError("e must be between 0 and 1")
    if abs(b) >= r1 + r2:
        raise ValueError("the discs miss each other: |b| must be below r1 + r2")
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)
    s = b / (r1 + r2)
    n = np.array([np.sqrt(1 - s * s), s])
    vn = float(np.dot(v1 - v2, n))
    if vn <= 0:
        raise ValueError("the discs are not approaching each other")
    return v1 - m2 / (m1 + m2) * (1 + e) * vn * n, v2 + m1 / (m1 + m2) * (1 + e) * vn * n


def ke_lost(m1, m2, v1, v2, b, r1, r2, e):
    """Kinetic energy lost: 1/2 mu (1 - e^2) v_n^2 with the reduced mass mu."""
    s = b / (r1 + r2)
    n = np.array([np.sqrt(1 - s * s), s])
    vn = float(np.dot(np.asarray(v1, float) - np.asarray(v2, float), n))
    return 0.5 * m1 * m2 / (m1 + m2) * (1 - e ** 2) * vn ** 2


def angle_deg(v):
    return float(np.degrees(np.arctan2(v[1], v[0])))

## Step 2: a contact simulation as an independent check
Instead of applying the formula, let the discs overlap a little and push each other with a stiff
spring force k x overlap along the line of centres (and a damper c x approach speed to lose energy).
We step the positions and velocities forward in small time steps and read the final velocities.
The result must come out close to the formula. For the damper we use the damping ratio
zeta = -ln(e) / sqrt(pi^2 + ln(e)^2), the textbook value for a linear spring and damper.

In [ ]:
def simulate_contact(m1, m2, v1, v2, b, r1, r2, e=1.0, k=1e6, dt=2e-6, gap=0.01):
    """Time-step two discs through a spring-damper contact. Returns (v1', v2')."""
    mu = m1 * m2 / (m1 + m2)
    zeta = 0.0 if e >= 1 else -np.log(e) / np.sqrt(np.pi ** 2 + np.log(e) ** 2)
    c = 2 * zeta * np.sqrt(k * mu)
    u1, u2 = np.asarray(v1, float).copy(), np.asarray(v2, float).copy()
    s = b / (r1 + r2)
    n0 = np.array([np.sqrt(1 - s * s), s])
    # start the discs a little way before contact, along the approach path
    x1 = np.zeros(2)
    x2 = (r1 + r2 + gap) * n0
    in_contact = False
    for _ in range(2_000_000):
        d = x2 - x1
        dist = float(np.hypot(d[0], d[1]))
        n = d / dist
        overlap = r1 + r2 - dist
        f = np.zeros(2)
        if overlap > 0:
            in_contact = True
            f_mag = max(k * overlap + c * float(np.dot(u1 - u2, n)), 0.0)   # the contact can only push
            f = -f_mag * n                                                   # force on disc 1
        elif in_contact and np.dot(u1 - u2, d) <= 0:
            break                                                            # separated and moving apart
        u1 = u1 + f / m1 * dt
        u2 = u2 - f / m2 * dt
        x1 = x1 + u1 * dt
        x2 = x2 + u2 * dt
    return u1, u2

## Step 3: a crash estimate with the work-energy idea
A car of mass m driving at speed v into a rigid wall stops in a crumple distance d. All its kinetic
energy is absorbed by the crumpling, so the average force and deceleration are

    F = m v^2 / (2 d)          a = v^2 / (2 d)

In [ ]:
def crash(mass, speed_kmh, crumple_m):
    v = speed_kmh / 3.6
    a = v ** 2 / (2 * crumple_m)
    return {"energy_kj": 0.5 * mass * v ** 2 / 1000, "decel": a, "decel_g": a / 9.81, "force_kn": mass * a / 1000}

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m1 = m2 = 1.0
    r1 = r2 = 0.5
    v1, v2 = np.array([4.0, 0.0]), np.zeros(2)
    a, b = disc_collision(m1, m2, v1, v2, 0.5, r1, r2, 1.0)
    print(f"b = 0.5 m, e = 1: disc 1 leaves with {a.round(3)} ({np.hypot(*a):.2f} m/s at {angle_deg(a):.1f} deg), "
          f"disc 2 with {b.round(3)} at {angle_deg(b):.1f} deg; angle between them {angle_deg(b) - angle_deg(a):.1f} deg")
    sa, sb = simulate_contact(m1, m2, v1, v2, 0.5, r1, r2, 1.0)
    print(f"simulation:   disc 1 {sa.round(3)}, disc 2 {sb.round(3)}")
    cr = crash(1200, 50, 0.6)
    print(f"car 1200 kg at 50 km/h, crumple 0.6 m: {cr['energy_kj']:.0f} kJ, {cr['decel_g']:.1f} g, average force {cr['force_kn']:.0f} kN")

    bs = np.linspace(0, 0.95, 96)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for e, col in ((1.0, "tab:blue"), (0.5, "tab:red")):
        ax.plot(bs, [np.hypot(*disc_collision(m1, m2, v1, v2, x, r1, r2, e)[0]) for x in bs], color=col, label=f"disc 1, e = {e}")
        ax.plot(bs, [np.hypot(*disc_collision(m1, m2, v1, v2, x, r1, r2, e)[1]) for x in bs], "--", color=col, label=f"disc 2, e = {e}")
    ax.set_xlabel("impact parameter b (m), R1 + R2 = 1 m")
    ax.set_ylabel("speed after the collision (m/s)")
    ax.set_title("A glancing hit hands over little speed")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "speeds_vs_impact.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    ax2.plot(bs, [angle_deg(disc_collision(m1, m2, v1, v2, x, r1, r2, 1.0)[0]) for x in bs], color="tab:blue", label="disc 1")
    ax2.plot(bs, [angle_deg(disc_collision(m1, m2, v1, v2, x, r1, r2, 1.0)[1]) for x in bs], color="tab:green", label="disc 2")
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_xlabel("impact parameter b (m), R1 + R2 = 1 m")
    ax2.set_ylabel("direction of motion after the collision (degrees)")
    ax2.set_title("Equal discs, e = 1: they leave at 90 degrees to each other")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "angles_vs_impact.png", dpi=120)

## Where this stops
The model has no friction between the discs (so no spin), no air drag and a perfectly stiff or perfectly
linear contact. A real car crash involves breaking structures, occupants and rotation. The roadmap adds
spin and many bodies.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
"""Checkpoint for Session 3. Prints PASS or FAIL."""
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    v1, v2 = np.array([4.0, 0.0]), np.zeros(2)
    a, b = L.disc_collision(1, 1, v1, v2, 0.5, 0.5, 0.5, 1.0)
    checks.append((f"worked example: disc 1 leaves with (1.00, -1.73) m/s (got {a.round(2)})", np.allclose(a, [1.0, -np.sqrt(3)])))
    checks.append((f"worked example: disc 2 leaves with (3.00, 1.73) m/s (got {b.round(2)})", np.allclose(b, [3.0, np.sqrt(3)])))
    checks.append(("worked example: the discs leave at 90 degrees to each other", abs(np.dot(a, b)) < 1e-12 and abs(L.angle_deg(b) - L.angle_deg(a) - 90) < 1e-9))
    cases = [(1, 1, 0.5, 1.0), (2, 1, 0.3, 0.6), (1, 3, 0.8, 0.0), (5, 2, 0.1, 0.9)]
    ok_p = ok_e = True
    for m1, m2, bb, e in cases:
        x, y = L.disc_collision(m1, m2, v1, v2, bb, 0.5, 0.5, e)
        ok_p &= np.allclose(m1 * x + m2 * y, m1 * v1 + m2 * v2)
        ke0 = 0.5 * m1 * v1 @ v1
        ok_e &= abs(ke0 - (0.5 * m1 * x @ x + 0.5 * m2 * y @ y) - L.ke_lost(m1, m2, v1, v2, bb, 0.5, 0.5, e)) < 1e-9
    checks.append(("the momentum vector of the pair is kept", ok_p))
    checks.append(("kinetic energy before - after equals the formula 1/2 mu (1 - e^2) v_n^2", ok_e))
    # the head-on case must agree with the one-dimensional formula of Session 2
    a0, b0 = L.disc_collision(2, 1, np.array([4.0, 0.0]), np.array([-1.0, 0.0]), 0.0, 0.5, 0.5, 0.5)
    checks.append(("head-on (b = 0) gives the one-dimensional answer of Session 2: 1.5 and 4.0 m/s", abs(a0[0] - 1.5) < 1e-9 and abs(b0[0] - 4.0) < 1e-9 and abs(a0[1]) < 1e-12))
    # the contact simulation is an independent check of the formula
    worst = 0.0
    for e in (1.0, 0.8, 0.5):
        fa, fb = L.disc_collision(1, 1, v1, v2, 0.5, 0.5, 0.5, e)
        sa, sb = L.simulate_contact(1, 1, v1, v2, 0.5, 0.5, 0.5, e)
        worst = max(worst, np.abs(fa - sa).max(), np.abs(fb - sb).max())
        checks.append((f"validation: the contact simulation conserves momentum for e = {e}", np.allclose(sa + sb, v1 + v2, atol=1e-9)))
    checks.append((f"validation: simulation and formula agree within 1.5 percent of the 4 m/s approach speed (worst {worst:.3f} m/s)", worst < 0.06))
    cr = L.crash(1200, 50, 0.6)
    checks.append((f"worked example: car 1200 kg, 50 km/h, 0.6 m crumple absorbs 116 kJ at 16.4 g and 193 kN (got {cr['energy_kj']:.0f} kJ, {cr['decel_g']:.1f} g, {cr['force_kn']:.0f} kN)",
                   abs(cr["energy_kj"] - 115.7) < 0.2 and abs(cr["decel_g"] - 16.4) < 0.05 and abs(cr["force_kn"] - 193) < 0.5))
    checks.append(("design: a crumple distance of 0.50 m keeps the average at or below 20 g, 0.49 m and 0.40 m do not",
                   L.crash(1200, 50, 0.50)["decel_g"] <= 20 < L.crash(1200, 50, 0.49)["decel_g"] < L.crash(1200, 50, 0.40)["decel_g"]))
    checks.append(("edge case: a grazing hit (b close to R1 + R2) hands over almost no speed", np.hypot(*L.disc_collision(1, 1, v1, v2, 0.999, 0.5, 0.5, 1.0)[1]) < 0.2))
    for bad, label in ((lambda: L.disc_collision(1, 1, v1, v2, 1.0, 0.5, 0.5, 1.0), "the discs miss (b = R1 + R2)"),
                       (lambda: L.disc_collision(1, 1, v1, v2, 0.2, 0.5, 0.5, 1.5), "e above 1"),
                       (lambda: L.disc_collision(1, 1, -v1, v2, 0.2, 0.5, 0.5, 1.0), "the discs moving apart"),
                       (lambda: L.disc_collision(0, 1, v1, v2, 0.2, 0.5, 0.5, 1.0), "mass 0")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    return checks
import types
L = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
ok = True
for label, passed in run_checks(L):
    print(('PASS  ' if passed else 'FAIL  ') + label)
    ok &= bool(passed)
print('PASS' if ok else 'FAIL')